In [1]:
# --- Instalações ---
# (O 'imblearn' é para o RandomUnderSampler)
!pip install -q scikit-learn pandas numpy tensorflow imbalanced-learn

# --- Imports de Sistema ---
import pandas as pd
import numpy as np
import sys
import os
import warnings

# --- Imports de Deep Learning (Keras/TensorFlow) ---
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Embedding, Dropout, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping # <-- Novo! Para combater overfitting

# --- Imports de Machine Learning (Sklearn) ---
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.exceptions import UndefinedMetricWarning

# --- Imports de Balanceamento ---
from imblearn.under_sampling import RandomUnderSampler

# Ignora warnings
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# --- Montar o Google Drive ---
print("Montando Google Drive...")
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print(f"Drive já montado ou erro: {e}")

# --- Confirmação de GPU ---
print("\nVerificando GPU...")
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  print(
      '\n\nATENÇÃO: GPU NÃO ENCONTRADA! '
      'Vá em "Ambiente de execução" -> "Alterar o tipo de ambiente de execução" e selecione "GPU (T4)".'
  )
else:
  print(f'GPU encontrada: {device_name}')

Montando Google Drive...
Mounted at /content/drive

Verificando GPU...
GPU encontrada: /device:GPU:0


In [7]:
# --- 1. Configurações ---

# --- ATENÇÃO: Verifique este caminho! ---
GDRIVE_PATH = '/content/drive/MyDrive/Faculdade/IC-2025.2/Biogenetica/dataset/'
ARQUIVO_ENTRADA = os.path.join(GDRIVE_PATH, 'dataset_FINAL_ACHATADO.csv')

TARGET_FUNCTION = 'protein binding'

# --- Configs da CNN ---
# Vamos "fatiar" todas as sequências nesse comprimento.
# 2000 bases é um bom equilíbrio entre manter o sinal e economizar RAM.
MAX_SEQ_LENGTH = 2000
EMBEDDING_DIM = 100  # Dimensão do vetor para cada base (A,T,C,G)

# --- Configs do Pipeline de Leitura ---
# Vamos ler o arquivo de 14.4M em chunks maiores para acelerar
CHUNK_SIZE = 100000

In [8]:
print(f"Iniciando Pipeline Final (CNN)...")
print(f"Arquivo de entrada: {ARQUIVO_ENTRADA}")
print(f"Tarefa: Classificação Binária (Target = '{TARGET_FUNCTION}')\n")

# --- 3.1 Carregar Dados (O "Motor RAM-Safe") ---
print(f"Carregando e achatando {ARQUIVO_ENTRADA} em pedaços (Chunks)...")
print("Isso VAI demorar alguns minutos. Não é um bug.")

colunas_necessarias = ['join_key', 'Sequencia', 'GO term name']
lista_de_labels_flat = []
lista_de_seqs_flat = []

try:
    reader = pd.read_csv(ARQUIVO_ENTRADA,
                         usecols=colunas_necessarias,
                         chunksize=CHUNK_SIZE,
                         low_memory=False) # Ajuda a evitar erros de tipo

    for i, chunk in enumerate(reader):
        print(f"Processando Pedaço (Chunk) {i}...")

        # 1. Prepara o label (Y) no chunk
        chunk['label'] = np.where(chunk['GO term name'] == TARGET_FUNCTION, 1, 0)

        # 2. Achata o chunk (Y)
        chunk_labels_flat = chunk.groupby('join_key')['label'].max()

        # 3. Achata o chunk (X)
        chunk_seq_flat = chunk.groupby('join_key')['Sequencia'].first()

        # 4. Guarda os resultados parciais
        lista_de_labels_flat.append(chunk_labels_flat)
        lista_de_seqs_flat.append(chunk_seq_flat)

    print("\nLeitura de chunks concluída. Consolidando dados...")

    # Consolida os resultados
    df_labels_parcial = pd.concat(lista_de_labels_flat)
    df_seqs_parcial = pd.concat(lista_de_seqs_flat)

    # Groupby FINAL
    print("Executando groupby final (Labels)...")
    df_labels_flat = df_labels_parcial.groupby(level=0).max()
    print("Executando groupby final (Sequências)...")
    df_seqs_flat = df_seqs_parcial.groupby(level=0).first()

    # Junta o X e Y finais
    df_flat = pd.merge(df_seqs_flat, df_labels_flat, left_index=True, right_index=True, how='inner')

    print(f"\nDataset achatado com sucesso para {len(df_flat)} genes únicos.")

except FileNotFoundError:
    print(f"ERRO: Arquivo '{ARQUIVO_ENTRADA}' não encontrado.")
    raise
except MemoryError:
    print("\n--- ERRO DE MEMÓRIA (Mesmo com Chunks) ---")
    print("O Colab (versão gratuita) pode não ter RAM suficiente para consolidar os ~80k genes.")
    print("Tente reiniciar o ambiente e rodar apenas esta célula.")
    raise

# --- 3.2 Balanceamento (Undersampling) ---
print(f"Distribuição ANTES do balanceamento:\n{df_flat['label'].value_counts()}\n")
print("Iniciando Undersampling para forçar balanço 1:1...")
rus = RandomUnderSampler(random_state=42)

X_para_amostrar = df_flat.index.values.reshape(-1, 1)
y_para_amostrar = df_flat['label']
X_res, y_res = rus.fit_resample(X_para_amostrar, y_para_amostrar)

indices_balanceados = X_res.flatten()
df_balanced = df_flat.loc[indices_balanceados].copy()
print(f"Dataset balanceado criado.")
print(f"Distribuição DEPOIS do balanceamento:\n{df_balanced['label'].value_counts()}\n")

# --- 3.3 Preparação para CNN (Tokenizer e Padding) ---
print("Preparando dados para CNN (Tokenizing e Padding)...")

# Garantir que as sequências são strings
df_balanced['Sequencia'] = df_balanced['Sequencia'].astype(str)

tokenizer = Tokenizer(char_level=True, lower=False, oov_token='U') # 'U' para desconhecido
tokenizer.fit_on_texts(df_balanced['Sequencia'])
sequences_tokenized = tokenizer.texts_to_sequences(df_balanced['Sequencia'])

# Atualiza o tamanho do vocabulário (A, T, C, G, N, U, etc.)
VOCAB_SIZE = len(tokenizer.word_index)
print(f"Tamanho do Vocabulário (A,T,C,G,N...): {VOCAB_SIZE}")

X_padded = pad_sequences(sequences_tokenized,
                         maxlen=MAX_SEQ_LENGTH,
                         padding='post',
                         truncating='post')

Y_labels = df_balanced['label'].values

print(f"Matriz de features X criada: {X_padded.shape}")
print(f"Vetor de labels Y criado: {Y_labels.shape}")

# --- 3.4 Divisão de Treino/Teste ---
print("\nDividindo dados (80% treino / 20% teste)...")
X_train, X_test, y_train, y_test = train_test_split(
    X_padded,
    Y_labels,
    test_size=0.2,
    random_state=42,
    stratify=Y_labels
)
print(f"Tamanho do Treino: {X_train.shape[0]}")
print(f"Tamanho do Teste: {X_test.shape[0]}")

Iniciando Pipeline Final (CNN)...
Arquivo de entrada: /content/drive/MyDrive/Faculdade/IC-2025.2/Biogenetica/dataset/dataset_FINAL_ACHATADO.csv
Tarefa: Classificação Binária (Target = 'protein binding')

Carregando e achatando /content/drive/MyDrive/Faculdade/IC-2025.2/Biogenetica/dataset/dataset_FINAL_ACHATADO.csv em pedaços (Chunks)...
Isso VAI demorar alguns minutos. Não é um bug.


ValueError: Usecols do not match columns, columns expected but not found: ['GO term name']

In [4]:
print("Construindo o modelo CNN (Otimizado contra Overfitting)...")

# Usamos +1 no VOCAB_SIZE porque 0 é reservado para o 'padding'
model = Sequential()

# 1. Camada de Embedding
model.add(Embedding(input_dim=VOCAB_SIZE + 1,
                    output_dim=EMBEDDING_DIM,
                    input_length=MAX_SEQ_LENGTH))

# 2. Bloco Convolucional 1
model.add(Conv1D(filters=64, kernel_size=10, activation='relu', padding='same'))
model.add(MaxPooling1D(pool_size=4))

# 3. Bloco Convolucional 2 (Mais profundo)
model.add(Conv1D(filters=128, kernel_size=8, activation='relu', padding='same'))
model.add(MaxPooling1D(pool_size=4))

# 4. Achatamento
# (Usar GlobalMaxPooling é muitas vezes melhor que Flatten para texto)
model.add(GlobalMaxPooling1D())

# 5. Camada Densa (Classificador)
model.add(Dense(256, activation='relu'))
# --- OTIMIZAÇÃO ANTI-OVERFITTING ---
model.add(Dropout(0.5)) # Desliga 50% dos neurônios no treino

# 6. Camada de Saída
model.add(Dense(1, activation='sigmoid'))

# Compila o modelo
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.summary()

Construindo o modelo CNN (Otimizado contra Overfitting)...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
print("\nIniciando treinamento da CNN FINAL...")
print("Usando EarlyStopping: o treino vai parar se a 'val_loss' não melhorar.")

# --- OTIMIZAÇÃO ANTI-OVERFITTING ---
# Se a perda na validação (val_loss) não melhorar por 3 épocas seguidas, pare.
early_stopping = EarlyStopping(monitor='val_loss',
                              patience=3,
                              restore_best_weights=True)

# Vamos treinar (Isso VAI demorar. Vá tomar um café de verdade)
history = model.fit(X_train, y_train,
                    epochs=20, # Máximo de 20, mas o EarlyStopping deve parar antes
                    batch_size=64,
                    validation_data=(X_test, y_test),
                    callbacks=[early_stopping]) # <-- Adiciona o callback

print("Treinamento concluído.")


Iniciando treinamento da CNN FINAL...
Usando EarlyStopping: o treino vai parar se a 'val_loss' não melhorar.
Epoch 1/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 17s 187ms/step - accuracy: 0.4670 - loss: 0.6953 - val_accuracy: 0.5725 - val_loss: 0.6919
Epoch 2/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.4976 - loss: 0.6930 - val_accuracy: 0.4990 - val_loss: 0.6916
Epoch 3/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.5117 - loss: 0.6927 - val_accuracy: 0.5706 - val_loss: 0.6874
Epoch 4/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.5328 - loss: 0.6908 - val_accuracy: 0.5106 - val_loss: 0.6927
Epoch 5/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.5056 - loss: 0.6961 - val_accuracy: 0.4990 - val_loss: 0.6922
Epoch 6/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.5125 - loss: 0.6928 - val_accuracy: 0.5048 - val_loss: 0.6896
Treinamento concluído.


In [6]:
print("\nAvaliando modelo final nos dados de teste...")

y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
print(f"\n--- Relatório de Desempenho (CNN FINAL) ---")
print(f"Acurácia Geral: {acc * 100:.2f}%")

print("\nRelatório de Classificação (Precisão, Recall, F1 por Classe):")
print(classification_report(y_test, y_pred))


Avaliando modelo final nos dados de teste...
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step

--- Relatório de Desempenho (CNN FINAL) ---
Acurácia Geral: 57.06%

Relatório de Classificação (Precisão, Recall, F1 por Classe):
              precision    recall  f1-score   support

           0       0.55      0.83      0.66       259
           1       0.65      0.31      0.42       258

    accuracy                           0.57       517
   macro avg       0.60      0.57      0.54       517
weighted avg       0.60      0.57      0.54       517

